# Домашнее задание 9

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42

### Загрузка данных с Kaggle

In [ ]:
sk_data = load_breast_cancer(as_frame=True)
X = sk_data.frame.drop(columns=["target"]).copy()
y = (sk_data.frame["target"] == 0).astype(int)
source_name = "sklearn.datasets.load_breast_cancer"

feature_cols = X.columns.tolist()
df = X.copy()
df["target"] = y.values
df["target_label"] = np.where(df["target"] == 1, "Malignant", "Benign")

print(f"Data source: {source_name}")
print(f"Dataset shape: {df.shape}")
display(df.head())
display(df["target_label"].value_counts().rename("count"))

## Часть 1. Анализ и очистка данных

In [ ]:
stats_df = df[feature_cols].describe().T
stats_df["median"] = df[feature_cols].median()

display(stats_df[["mean", "std", "min", "25%", "50%", "median", "75%", "max"]].round(2))

In [ ]:
n_features = len(feature_cols)
n_cols = 3
n_rows = int(np.ceil(n_features / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 4.2 * n_rows))
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    sns.histplot(
        data=df,
        x=col,
        hue="target_label",
        kde=True,
        stat="density",
        common_norm=False,
        element="step",
        alpha=0.3,
        ax=axes[i],
    )
    axes[i].set_title(col)
    axes[i].legend_.set_title("Class")

for j in range(i + 1, len(axes)):
    axes[j].axis("off")

plt.suptitle("Распределения признаков по классам", y=1.01, fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
corr = df[feature_cols].corr()

plt.figure(figsize=(16, 12))
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("Heatmap корреляционной матрицы")
plt.show()

threshold = 0.85
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
strong_pairs = (
    upper.stack()
    .reset_index(name="corr")
    .rename(columns={"level_0": "feature_1", "level_1": "feature_2"})
)
strong_pairs["abs_corr"] = strong_pairs["corr"].abs()
strong_pairs = strong_pairs[strong_pairs["abs_corr"] > threshold].sort_values("abs_corr", ascending=False)

print(f"Количество пар с |corr| > {threshold}: {len(strong_pairs)}")
display(strong_pairs.head(20))

In [ ]:
top_pairs = strong_pairs.head(6)

if len(top_pairs) == 0:
    print("Сильно скоррелированных пар по выбранному порогу не найдено.")
else:
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()

    for ax, (_, row) in zip(axes, top_pairs.iterrows()):
        sns.scatterplot(
            data=df,
            x=row["feature_1"],
            y=row["feature_2"],
            hue="target_label",
            alpha=0.75,
            s=25,
            ax=ax,
        )
        ax.set_title(f"corr={row['corr']:.3f}: {row['feature_1']} vs {row['feature_2']}")

    for idx in range(len(top_pairs), len(axes)):
        axes[idx].axis("off")

    plt.suptitle("Scatterplot для сильно коррелированных признаков", y=1.02, fontsize=16)
    plt.tight_layout()
    plt.show()

In [ ]:
# Отбираем признаки, где различие между классами наиболее выражено.
class_means = df.groupby("target")[feature_cols].mean().T
effect_size = ((class_means[1] - class_means[0]).abs() / (df[feature_cols].std() + 1e-9)).sort_values(ascending=False)
top_box_features = effect_size.head(10).index.tolist()

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

for ax, col in zip(axes, top_box_features):
    sns.boxplot(data=df, x="target_label", y=col, ax=ax)
    ax.set_title(col)
    ax.set_xlabel("")

plt.suptitle("Boxplot для признаков с максимальным разделением классов", y=1.02, fontsize=16)
plt.tight_layout()
plt.show()

print("Топ-10 наиболее разделяющих признаков (по грубой нормированной разнице средних):")
display(effect_size.head(10).rename("effect_size"))

## Часть 2. Моделирование при помощи kNN

Стандартизация обязательна для kNN, потому что метод использует расстояния между объектами. Если признаки измеряются в разных масштабах, признаки с большими численными значениями доминируют в метрике расстояния и искажают поиск соседей.

In [ ]:
X_full = df[feature_cols].copy()
y_full = df["target"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X_full,
    y_full,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y_full,
)

print(f"Train size: {X_train.shape}, Test size: {X_test.shape}")


def evaluate_binary_model(model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_test)[:, 1]
    else:
        y_score = model.decision_function(X_test)

    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1_score": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_score),
    }
    fpr, tpr, _ = roc_curve(y_test, y_score)

    return metrics, fpr, tpr

In [ ]:
knn_baseline = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier()),
    ]
)

knn_base_metrics, knn_base_fpr, knn_base_tpr = evaluate_binary_model(
    knn_baseline, X_train, y_train, X_test, y_test
)

display(pd.DataFrame([knn_base_metrics], index=["kNN baseline"]).round(4))

plt.figure(figsize=(7, 5))
plt.plot(knn_base_fpr, knn_base_tpr, label=f"kNN baseline (AUC={knn_base_metrics['roc_auc']:.3f})")
plt.plot([0, 1], [0, 1], "k--", alpha=0.6)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC-кривая: kNN baseline")
plt.legend()
plt.show()

In [ ]:
knn_grid = {
    "knn__n_neighbors": list(range(3, 32, 2)),
    "knn__weights": ["uniform", "distance"],
    "knn__p": [1, 2],
}

knn_search = GridSearchCV(
    estimator=knn_baseline,
    param_grid=knn_grid,
    scoring="roc_auc",
    cv=5,
    n_jobs=-1,
)
knn_search.fit(X_train, y_train)

best_knn = knn_search.best_estimator_
knn_tuned_metrics, knn_tuned_fpr, knn_tuned_tpr = evaluate_binary_model(
    best_knn, X_train, y_train, X_test, y_test
)

print("Лучшие параметры kNN:", knn_search.best_params_)
print(f"Лучший CV ROC-AUC: {knn_search.best_score_:.4f}")

knn_compare_df = pd.DataFrame(
    [knn_base_metrics, knn_tuned_metrics],
    index=["kNN baseline", "kNN tuned"],
).round(4)
display(knn_compare_df)

plt.figure(figsize=(7, 5))
plt.plot(knn_base_fpr, knn_base_tpr, label=f"kNN baseline (AUC={knn_base_metrics['roc_auc']:.3f})")
plt.plot(knn_tuned_fpr, knn_tuned_tpr, label=f"kNN tuned (AUC={knn_tuned_metrics['roc_auc']:.3f})")
plt.plot([0, 1], [0, 1], "k--", alpha=0.6)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC-кривые: kNN baseline vs tuned")
plt.legend()
plt.show()

## Бонус: логистическая регрессия

Сначала удаляем признаки, имеющие парную корреляцию Пирсона выше 0.85.

In [ ]:
corr_abs = X_full.corr().abs()
upper_abs = corr_abs.where(np.triu(np.ones(corr_abs.shape), k=1).astype(bool))
to_drop = [col for col in upper_abs.columns if (upper_abs[col] > 0.85).any()]

X_reduced = X_full.drop(columns=to_drop)

print(f"Исходное число признаков: {X_full.shape[1]}")
print(f"Удалено из-за корреляции > 0.85: {len(to_drop)}")
print(f"Осталось признаков: {X_reduced.shape[1]}")
if to_drop:
    display(pd.Series(to_drop, name="dropped_features"))

In [ ]:
Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X_reduced,
    y_full,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y_full,
)

logreg_baseline = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("logreg", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)),
    ]
)

log_base_metrics, log_base_fpr, log_base_tpr = evaluate_binary_model(
    logreg_baseline, Xr_train, yr_train, Xr_test, yr_test
)

display(pd.DataFrame([log_base_metrics], index=["LogReg baseline"]).round(4))

coef_base = pd.Series(
    logreg_baseline.named_steps["logreg"].coef_.ravel(),
    index=X_reduced.columns,
).sort_values()

plt.figure(figsize=(10, 8))
coef_base.plot(kind="barh")
plt.title("Коэффициенты baseline LogisticRegression")
plt.xlabel("Coefficient value")
plt.show()

plt.figure(figsize=(7, 5))
plt.plot(log_base_fpr, log_base_tpr, label=f"LogReg baseline (AUC={log_base_metrics['roc_auc']:.3f})")
plt.plot([0, 1], [0, 1], "k--", alpha=0.6)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC-кривая: LogisticRegression baseline")
plt.legend()
plt.show()

In [ ]:
logreg_cv = Pipeline(
    [
        ("scaler", StandardScaler()),
        (
            "logreg_cv",
            LogisticRegressionCV(
                Cs=np.logspace(-3, 3, 25),
                cv=5,
                scoring="roc_auc",
                max_iter=5000,
                solver="liblinear",
                penalty="l2",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            ),
        ),
    ]
)

log_cv_metrics, log_cv_fpr, log_cv_tpr = evaluate_binary_model(
    logreg_cv, Xr_train, yr_train, Xr_test, yr_test
)

best_c = float(logreg_cv.named_steps["logreg_cv"].C_[0])
print(f"Лучшее значение C по кросс-валидации: {best_c:.6f}")

log_compare_df = pd.DataFrame(
    [log_base_metrics, log_cv_metrics],
    index=["LogReg baseline", "LogReg tuned (CV)"],
).round(4)
display(log_compare_df)

coef_cv = pd.Series(
    logreg_cv.named_steps["logreg_cv"].coef_.ravel(),
    index=X_reduced.columns,
).sort_values()

plt.figure(figsize=(10, 8))
coef_cv.plot(kind="barh")
plt.title("Коэффициенты LogisticRegressionCV")
plt.xlabel("Coefficient value")
plt.show()

plt.figure(figsize=(7, 5))
plt.plot(log_base_fpr, log_base_tpr, label=f"LogReg baseline (AUC={log_base_metrics['roc_auc']:.3f})")
plt.plot(log_cv_fpr, log_cv_tpr, label=f"LogReg tuned (AUC={log_cv_metrics['roc_auc']:.3f})")
plt.plot([0, 1], [0, 1], "k--", alpha=0.6)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC-кривые: LogisticRegression baseline vs tuned")
plt.legend()
plt.show()

In [ ]:
final_compare = pd.DataFrame(
    [knn_tuned_metrics, log_cv_metrics],
    index=["kNN tuned", "LogReg tuned (CV)"],
).round(4)

display(final_compare)

best_by_auc = final_compare["roc_auc"].idxmax()
best_by_f1 = final_compare["f1_score"].idxmax()

print(f"Лучшая модель по ROC-AUC: {best_by_auc}")
print(f"Лучшая модель по F1-score: {best_by_f1}")